# Tensors Masterclass: The Foundation of PyTorch

**What you'll learn in this notebook:**
- What tensors are and how they generalize scalars, vectors, and matrices
- Every way to create tensors in PyTorch
- Tensor properties: shape, dtype, device, memory layout
- Choosing the right dtype for your use case
- Arithmetic operations and reductions
- Reshaping, transposing, and permuting
- Indexing: basic, boolean, and fancy
- Broadcasting rules demystified
- Views vs copies and memory semantics

**Prerequisites:** Basic Python and NumPy familiarity  
**Time:** ~45 minutes  
**Hardware:** CPU only — no GPU required!

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("We'll use CPU for everything in this notebook.")

## 1. What is a Tensor?

A **tensor** is a generalization of scalars, vectors, and matrices to arbitrary dimensions:

| Rank | Name | Shape Example | Real-world analogy |
|------|--------|---------------|-------------------|
| 0 | Scalar | `()` | A single temperature reading |
| 1 | Vector | `(3,)` | RGB color values |
| 2 | Matrix | `(3, 4)` | A grayscale image |
| 3 | 3D Tensor | `(3, 224, 224)` | An RGB image (C, H, W) |
| 4 | 4D Tensor | `(32, 3, 224, 224)` | A batch of RGB images |

Let's create one of each and inspect them:

In [ ]:
scalar = torch.tensor(3.14)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])
tensor_3d = torch.randn(3, 4, 5)

for name, t in [("Scalar", scalar), ("Vector", vector), ("Matrix", matrix), ("3D Tensor", tensor_3d)]:
    print(f"{name:10s} | ndim={t.ndim} | shape={str(t.shape):15s} | numel={t.numel()}")
print(f"\nScalar value: {scalar.item()}")
print(f"Vector: {vector}")

## 2. Creating Tensors

PyTorch provides many factory functions. Here's the complete toolkit:

In [ ]:
print("=== Zeros, Ones, and Constants ===")
print(f"zeros(2,3):\n{torch.zeros(2, 3)}\n")
print(f"ones(2,3):\n{torch.ones(2, 3)}\n")
print(f"full(2,3, fill=7):\n{torch.full((2, 3), 7.0)}\n")

print("=== Identity and Diagonal ===")
print(f"eye(3):\n{torch.eye(3)}\n")
print(f"diag([1,2,3]):\n{torch.diag(torch.tensor([1, 2, 3]))}\n")

print("=== Sequences ===")
print(f"arange(0, 10, 2): {torch.arange(0, 10, 2)}")
print(f"linspace(0, 1, 5): {torch.linspace(0, 1, 5)}")

In [ ]:
print("=== Random Tensors ===")
torch.manual_seed(42)
print(f"randn(2,3) — normal(0,1):\n{torch.randn(2, 3)}\n")
print(f"rand(2,3) — uniform[0,1):\n{torch.rand(2, 3)}\n")
print(f"randint(0,10, (2,3)):\n{torch.randint(0, 10, (2, 3))}\n")

print("=== From existing data ===")
import numpy as np
np_array = np.array([1.0, 2.0, 3.0])
from_numpy = torch.from_numpy(np_array)
from_list = torch.tensor([1.0, 2.0, 3.0])
print(f"from_numpy: {from_numpy}")
print(f"from list:  {from_list}")

print("\n=== *_like functions (match shape & dtype) ===")
x = torch.randn(2, 3)
print(f"zeros_like(x): {torch.zeros_like(x).shape}")
print(f"ones_like(x):  {torch.ones_like(x).shape}")
print(f"randn_like(x): {torch.randn_like(x).shape}")

## 3. Tensor Properties

Every tensor carries metadata about its shape, data type, device, and memory layout:

In [ ]:
x = torch.randn(3, 4, 5)

print(f"shape:       {x.shape}")
print(f"size():      {x.size()}")        # same as .shape
print(f"ndim:        {x.ndim}")           # number of dimensions
print(f"numel():     {x.numel()}")        # total number of elements
print(f"dtype:       {x.dtype}")          # data type
print(f"device:      {x.device}")         # cpu or cuda:N
print(f"is_floating: {x.is_floating_point()}")
print(f"stride:      {x.stride()}")       # memory stride per dimension
print(f"is_contig:   {x.is_contiguous()}") # contiguous in memory?

## 4. Dtype Overview — Practical Guide

Choosing the right dtype affects **precision**, **memory**, and **speed**:

| Dtype | Bits | Range | Use case |
|-------|------|-------|----------|
| `float32` | 32 | ~7 decimal digits | Default for training |
| `float64` | 64 | ~15 decimal digits | Scientific computing, rare in DL |
| `float16` | 16 | ~3 decimal digits | Mixed precision inference |
| `bfloat16` | 16 | ~3 digits, larger range | Mixed precision training (preferred) |
| `int64` | 64 | ±9.2×10¹⁸ | Indices, labels |
| `int32` | 32 | ±2.1×10⁹ | Indices (memory-efficient) |
| `bool` | 8 | True/False | Masks |

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
print(f"Default dtype: {x.dtype}")

for dtype in [torch.float16, torch.bfloat16, torch.float32, torch.float64]:
    t = x.to(dtype)
    bytes_per_elem = t.element_size()
    print(f"  {str(dtype):20s} | {bytes_per_elem} bytes/elem | total: {t.numel() * bytes_per_elem:2d} bytes | value: {t[0]}")

print("\n--- Precision comparison ---")
val = torch.tensor(1.0) + torch.tensor(1e-7)
print(f"float32: 1.0 + 1e-7 = {val.item():.10f}")
val16 = torch.tensor(1.0, dtype=torch.float16) + torch.tensor(1e-7, dtype=torch.float16)
print(f"float16: 1.0 + 1e-7 = {val16.item():.10f}  (precision lost!)")
valbf = torch.tensor(1.0, dtype=torch.bfloat16) + torch.tensor(1e-7, dtype=torch.bfloat16)
print(f"bfloat16: 1.0 + 1e-7 = {valbf.item():.10f} (also lost, but bfloat16 has larger range)")

## 5. Basic Operations

PyTorch supports element-wise arithmetic, matrix multiplication, and reductions.
All operations return new tensors (unless using in-place `_` variants).

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("=== Element-wise arithmetic ===")
print(f"a + b     = {a + b}")
print(f"a * b     = {a * b}")
print(f"a ** 2    = {a ** 2}")
print(f"torch.sqrt(a) = {torch.sqrt(a)}")

print("\n=== Dot product and matrix multiply ===")
print(f"a @ b (dot product) = {a @ b}")
A = torch.randn(2, 3)
B = torch.randn(3, 4)
print(f"A{list(A.shape)} @ B{list(B.shape)} = shape {list((A @ B).shape)}")

print("\n=== Comparison (returns bool tensors) ===")
print(f"a > 2:  {a > 2}")
print(f"a == 2: {a == 2}")

### Reductions with the `dim` parameter

The `dim` argument specifies **which dimension to collapse**. Think of it as:
"reduce along this axis, removing it from the output shape."

For a tensor of shape `(3, 4)`:
- `dim=0` → collapse rows → result shape `(4,)`
- `dim=1` → collapse columns → result shape `(3,)`

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0]])
print(f"x (shape {list(x.shape)}):\n{x}\n")

print(f"sum()            = {x.sum()}")
print(f"sum(dim=0)       = {x.sum(dim=0)}  shape: {list(x.sum(dim=0).shape)} (collapsed rows)")
print(f"sum(dim=1)       = {x.sum(dim=1)}  shape: {list(x.sum(dim=1).shape)} (collapsed cols)")
print(f"sum(dim=1, keepdim=True):\n{x.sum(dim=1, keepdim=True)}  shape: {list(x.sum(dim=1, keepdim=True).shape)}")

print(f"\nmean(dim=1)      = {x.mean(dim=1)}")
print(f"max(dim=1).values = {x.max(dim=1).values}")
print(f"argmax(dim=1)     = {x.argmax(dim=1)}")

## 6. Reshaping Tensors

PyTorch provides several ways to change tensor shapes. The key distinction is whether the operation creates a **view** (shares memory) or a **copy** (new memory).

| Function | Creates View? | Notes |
|----------|:---:|-------|
| `view()` | Yes | Requires contiguous memory |
| `reshape()` | Maybe | Returns view if possible, copy otherwise |
| `flatten()` | Maybe | Shorthand for `reshape(-1)` |
| `transpose()` | Yes | Swaps two dims |
| `permute()` | Yes | Arbitrary dim reordering |
| `unsqueeze()` | Yes | Adds dim of size 1 |
| `squeeze()` | Yes | Removes dims of size 1 |

In [ ]:
x = torch.arange(12)
print(f"Original: shape={list(x.shape)}, data={x}")

reshaped = x.view(3, 4)
print(f"\nview(3,4):\n{reshaped}  shape={list(reshaped.shape)}")

reshaped2 = x.view(2, 2, 3)
print(f"\nview(2,2,3):\n{reshaped2}  shape={list(reshaped2.shape)}")

flat = reshaped.flatten()
print(f"\nflatten(): {flat}  shape={list(flat.shape)}")

print(f"\nreshape(4, -1): shape={list(x.reshape(4, -1).shape)} (-1 is inferred)")

print("\n--- unsqueeze and squeeze ---")
v = torch.tensor([1, 2, 3])
print(f"v:                shape={list(v.shape)}")
print(f"unsqueeze(0):     shape={list(v.unsqueeze(0).shape)}  (add batch dim)")
print(f"unsqueeze(1):     shape={list(v.unsqueeze(1).shape)}  (add column dim)")
u = torch.randn(1, 3, 1)
print(f"squeeze({list(u.shape)}): shape={list(u.squeeze().shape)}")

In [ ]:
print("=== Transpose and Permute ===")
x = torch.randn(2, 3, 4)
print(f"Original shape:            {list(x.shape)}")
print(f"transpose(0, 2) shape:     {list(x.transpose(0, 2).shape)}")
print(f"permute(2, 0, 1) shape:    {list(x.permute(2, 0, 1).shape)}")

print("\n--- Common pattern: (B, C, H, W) -> (B, H, W, C) ---")
images = torch.randn(8, 3, 224, 224)
print(f"NCHW: {list(images.shape)}")
nhwc = images.permute(0, 2, 3, 1)
print(f"NHWC: {list(nhwc.shape)}")

## 7. Indexing

PyTorch supports NumPy-style indexing: basic slicing, boolean masks, and fancy (integer array) indexing.

In [ ]:
x = torch.arange(20).reshape(4, 5).float()
print(f"x:\n{x}\n")

print("=== Basic indexing ===")
print(f"x[0]       = {x[0]}          (first row)")
print(f"x[1, 3]    = {x[1, 3]}            (row 1, col 3)")
print(f"x[:, 0]    = {x[:, 0]}   (first column)")
print(f"x[1:3]     =\n{x[1:3]}    (rows 1-2)")
print(f"x[::2]     =\n{x[::2]}    (every other row)")

print("\n=== Boolean indexing ===")
mask = x > 10
print(f"x > 10:\n{mask}")
print(f"x[x > 10] = {x[mask]}  (flat result)")

print("\n=== Fancy indexing (index with integer tensors) ===")
rows = torch.tensor([0, 2, 3])
print(f"x[{rows.tolist()}] =\n{x[rows]}")
cols = torch.tensor([1, 3, 4])
print(f"x[{rows.tolist()}, {cols.tolist()}] = {x[rows, cols]}  (pairs: (0,1), (2,3), (3,4))")

## 8. Broadcasting

Broadcasting lets you do operations on tensors of **different shapes** without explicitly copying data.

**Broadcasting rules** (checked right to left):
1. Dimensions are compatible if they are equal, or one of them is 1
2. Missing dimensions are treated as size 1 (prepended on the left)

Example: `(3, 4) + (4,)` works because `(4,)` becomes `(1, 4)`, and dim 0 broadcasts `1 → 3`.

In [ ]:
print("=== Scalar broadcast ===")
x = torch.tensor([1.0, 2.0, 3.0])
print(f"[1,2,3] + 10 = {x + 10}\n")

print("=== Row + Column broadcast ===")
row = torch.tensor([[1, 2, 3]])       # shape (1, 3)
col = torch.tensor([[10], [20], [30]])  # shape (3, 1)
result = row + col                      # shape (3, 3)
print(f"  row {list(row.shape)}: {row}")
print(f"+ col {list(col.shape)}: {col.squeeze()}")
print(f"= result {list(result.shape)}:\n{result}\n")

print("=== Step-by-step shape analysis ===")
a = torch.randn(8, 1, 6, 1)
b = torch.randn(   7, 1, 5)
c = a + b
print(f"  a shape: {list(a.shape):>16s}")
print(f"  b shape: {list(b.shape):>16s}  (prepend 1 -> [1, 7, 1, 5])")
print(f"  result:  {list(c.shape):>16s}  (broadcast each dim)")

print("\n=== Common pitfall: mean subtraction ===")
data = torch.randn(3, 4)
row_means = data.mean(dim=1, keepdim=True)  # keepdim=True is essential!
centered = data - row_means
print(f"data {list(data.shape)} - means {list(row_means.shape)} = centered {list(centered.shape)}")
print(f"Row means after centering: {centered.mean(dim=1)}")  # ~0

## 9. Views vs Copies

A **view** shares the same underlying memory as the original tensor — modifying one modifies both. A **copy** has its own independent memory.

Key rule: reshaping operations (view, transpose, slice) create views. Operations that change data layout (like indexing with a tensor) create copies.

In [ ]:
x = torch.arange(6).float()
y = x.view(2, 3)  # y is a VIEW of x

print(f"x = {x}")
print(f"y = x.view(2,3):\n{y}")
print(f"Same memory? data_ptr equal: {x.data_ptr() == y.data_ptr()}")

y[0, 0] = 999
print(f"\nAfter y[0,0] = 999:")
print(f"x = {x}  <-- x changed too! (shared memory)")
print(f"y:\n{y}")

print("\n--- Now with a copy ---")
x = torch.arange(6).float()
z = x.clone()  # z is an independent COPY
z[0] = 999
print(f"x = {x}  <-- x unchanged (independent copy)")
print(f"z = {z}")

print("\n--- Boolean indexing creates a copy ---")
x = torch.arange(6).float()
selected = x[x > 3]
print(f"Same memory? {x.data_ptr() == selected.data_ptr()}")  # False

## 10. Contiguity and Stride

When you transpose a tensor, the data stays in the same memory — only the **strides** change. This can make the tensor non-contiguous, which matters for some operations.

In [ ]:
x = torch.arange(6).reshape(2, 3)
print(f"x:\n{x}")
print(f"x strides: {x.stride()}, contiguous: {x.is_contiguous()}")

xt = x.t()
print(f"\nx.t():\n{xt}")
print(f"x.t() strides: {xt.stride()}, contiguous: {xt.is_contiguous()}")
print(f"Same memory? {x.data_ptr() == xt.data_ptr()}")

try:
    xt.view(-1)
except RuntimeError as e:
    print(f"\nxt.view(-1) fails: {e}")

print(f"xt.contiguous().view(-1) works: {xt.contiguous().view(-1)}")
print(f"xt.reshape(-1) also works:      {xt.reshape(-1)} (auto-contiguous)")

## 11. Concatenation and Stacking

Two essential ways to combine tensors:
- `cat` — join along an **existing** dimension
- `stack` — join along a **new** dimension

In [ ]:
a = torch.randn(2, 3)
b = torch.randn(2, 3)

print(f"a shape: {list(a.shape)}, b shape: {list(b.shape)}")
print(f"cat(dim=0) shape: {list(torch.cat([a, b], dim=0).shape)}  (stack vertically)")
print(f"cat(dim=1) shape: {list(torch.cat([a, b], dim=1).shape)}  (stack horizontally)")
print(f"stack(dim=0) shape: {list(torch.stack([a, b], dim=0).shape)}  (new batch dim)")
print(f"stack(dim=1) shape: {list(torch.stack([a, b], dim=1).shape)}  (interleave)")

print("\n--- Practical: batching sequences of different lengths ---")
seqs = [torch.randn(3), torch.randn(5), torch.randn(4)]
padded = torch.nn.utils.rnn.pad_sequence(
    [s.unsqueeze(1) for s in seqs], batch_first=True, padding_value=0.0
)
print(f"Padded batch shape: {list(padded.shape)}")

## 🧪 Try It Yourself: Row Normalization

**Exercise:** Create a function that normalizes each row of a matrix to unit length (L2 norm = 1).

Given a matrix of shape `(N, D)`, each row should have `||row||_2 = 1` after normalization.

Hint: Use `torch.norm()` or `torch.linalg.norm()` with `keepdim=True`.

In [ ]:
def normalize_rows(matrix):
    """Normalize each row of matrix to unit L2 norm."""
    # YOUR CODE HERE — replace the line below
    # Hint: compute row norms with keepdim=True, then divide
    norms = torch.linalg.norm(matrix, dim=1, keepdim=True)
    return matrix / norms

# Test it
torch.manual_seed(42)
M = torch.randn(4, 5)
M_normalized = normalize_rows(M)

print(f"Original matrix:\n{M}\n")
print(f"Normalized matrix:\n{M_normalized}\n")

row_norms = torch.linalg.norm(M_normalized, dim=1)
print(f"Row norms after normalization: {row_norms}")
print(f"All unit length? {torch.allclose(row_norms, torch.ones(4))}")

### Try modifying the exercise:

1. What happens if a row is all zeros? Add a small epsilon to prevent division by zero.
2. Normalize columns instead of rows.
3. Use `torch.nn.functional.normalize()` — how does it compare?

In [ ]:
# Try it yourself! Experiment here:
import torch.nn.functional as F

# Compare with F.normalize
M_fn = F.normalize(M, p=2, dim=1)
print(f"F.normalize matches our function? {torch.allclose(M_normalized, M_fn)}")

# Handle zero rows
M_with_zeros = M.clone()
M_with_zeros[0] = 0.0
print(f"\nMatrix with zero row:\n{M_with_zeros}")
safe_normalized = F.normalize(M_with_zeros, p=2, dim=1)
print(f"Safe normalized (zero row stays zero):\n{safe_normalized}")

## Key Takeaways

1. **Tensors are the core data structure** — everything in PyTorch flows through tensors
2. **Shape matters** — always track shapes mentally; print them when debugging
3. **Use `float32` by default** for training; `bfloat16` for mixed precision; `int64` for indices
4. **Reductions with `dim`** — the dim you specify gets collapsed (removed from the shape)
5. **`keepdim=True`** is essential for broadcasting after reductions
6. **Views share memory** — `view`, `reshape`, `transpose`, slicing are cheap but linked
7. **Broadcasting** — dimensions are matched right-to-left; size 1 expands to match
8. **Use `clone()`** when you need an independent copy
9. **`contiguous()` or `reshape()`** — needed after transpose before view operations

**Next up:** Notebook 02 — Autograd from Scratch